# Phase 1 資料下載與驗證測試

1. 請將 Jupyter 的工作目錄設為**專案根目錄**（含 `utils/`、`data_pipeline.py` 的那一层）。
2. 你已決定統一使用 `C:\\Data_Lake`；若你要改到其他磁碟，請在下方設定 `DATA_LAKE_ROOT`（例如 `E:\\Data_Lake`）。
3. **小範圍測試**使用 `session_tag="notebook"`，不會覆寫正式檔 `*_1m_2021-2026.parquet`，也不會與正式下載共用 checkpoint。

In [ ]:
import os
import sys

# 專案根目錄加入路徑（若 Notebook 啟動目錄已是根目錄可略過）
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# os.environ["DATA_LAKE_ROOT"] = r"C:\Data_Lake"
print("DATA_LAKE_ROOT =", os.environ.get("DATA_LAKE_ROOT", "(未設定，將使用預設 C:\\Data_Lake)"))

## 小範圍下載（約一週，僅測試連線與欄位）

In [ ]:
from utils.binance_downloader import download_binance_klines
from utils.config import final_parquet_path
from utils.data_validator import clean_klines, validate_klines

df_raw = download_binance_klines(
    "BTCUSDT",
    start_date="2024-01-01",
    end_date="2024-01-07",
    force_redownload=False,
    session_tag="notebook",
)
df_clean = clean_klines(
    df_raw,
    "BTCUSDT",
    reindex_start_date="2024-01-01",
    reindex_end_date="2024-01-07",
)
out = final_parquet_path("BTCUSDT", "1m", date_range_label="notebook")
df_clean.to_parquet(out, index=True)
df_clean.head()

## 讀取測試產物並驗證（檔名標籤為 `notebook`）

In [ ]:
from utils.data_loader import load_raw_klines

df = load_raw_klines("BTCUSDT", "1m", "notebook")
display(df.head())
validate_klines("BTCUSDT", date_range="notebook", use_project_expected_length=False)

## 正式完整下載

請在終端機執行：`python data_pipeline.py`（資料量大，不建議在 Notebook 長時間執行）。
完成後可將上一儲存格的 `date_range` 改為 `"2021-2026"` 並重新執行驗證。